# M9 · LandLaw AI — Hỏi đáp bằng giọng nói

Chạy trên Google Colab với GPU T4. Demo dùng Hybrid RAG, Qwen 4-bit, nhận dạng tiếng Việt và Edge TTS tùy chọn. Giao diện gồm hội thoại, câu hỏi gợi ý và nguồn trích dẫn.

Chạy các cell lần lượt. Nếu chỉ sửa giao diện, chạy lại cell cuối để giữ model đã nạp. Link Gradio dùng được trong thời gian runtime còn hoạt động; người có link có thể truy cập demo.


In [1]:
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive')
# Sửa nếu bạn lưu project ở vị trí khác trên Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa')
assert (PROJECT_ROOT / 'src/landlaw_rag/audio/demo.py').is_file(), f'Không tìm thấy project: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))


Mounted at /content/drive


In [2]:
%pip install -q -r requirements/audio.txt

# Nếu Colab yêu cầu khởi động lại runtime sau cài đặt, khởi động lại rồi chạy lại cell Drive.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 12.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 22.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 20.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 86.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.7 

In [3]:
import torch
import gradio as gr

print('Gradio:', gr.__version__)
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
required_files = ['data/processed/chunks.json', 'indexes/bm25/bm25_index.json.gz']
for relative_path in required_files:
    assert (PROJECT_ROOT / relative_path).is_file(), f'Thiếu dữ liệu: {relative_path}; chạy các milestone xây index trước.'
assert (PROJECT_ROOT / 'indexes/faiss').is_dir(), 'Thiếu indexes/faiss; chạy milestone vector index trước.'


Gradio: 5.49.1
GPU: Tesla T4


In [4]:
from landlaw_rag.audio.runtime import load_colab_rag
from landlaw_rag.audio import AudioInteraction, WhisperSTT, EdgeTTS

# Chỉ nạp model một lần; không chạy lại cell này khi chỉ cập nhật UI.
rag = load_colab_rag(PROJECT_ROOT, model_name='Qwen/Qwen3-4B-Instruct-2507', strict_citations=False)
interaction = AudioInteraction(
    rag,
    stt=WhisperSTT(
        model_size='vinai/PhoWhisper-medium',
        device='cuda',
        compute_type='float16',
        backend='transformers',
    ),
    tts=EdgeTTS(voice='vi-VN-HoaiMyNeural'),
    output_dir='/tmp/m9_audio',
)
print('Pipeline đã nạp. Tiếp tục chạy cell mở demo.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Pipeline đã nạp. Tiếp tục chạy cell mở demo.


## Mở / cập nhật giao diện

Cell dưới tải lại `demo.py` từ Drive và đóng giao diện cũ trước khi mở bản mới. Không cần nạp lại Qwen khi sửa UI.

Bạn có thể dùng demo ngay trong notebook hoặc mở link `gradio.live` để có không gian rộng hơn và sử dụng microphone. Thu âm → nhận dạng → kiểm tra/sửa câu hỏi → gửi. Checkbox đọc câu trả lời mặc định tắt; bật lên sẽ gửi nội dung trả lời tới Edge TTS trực tuyến. Audio được lưu tạm trong `/tmp/m9_audio` và mất khi runtime bị xóa.

Lịch sử hiển thị trong phiên; mỗi câu hỏi vào RAG được xử lý độc lập. Muốn đối chiếu, nêu rõ “so sánh Luật Đất đai 2013 và hiện hành” trong câu hỏi.


In [7]:
import importlib
import landlaw_rag.audio.demo as demo_module

importlib.reload(demo_module)

try:
    demo.close()
except Exception:
    pass

demo = demo_module.build_demo(interaction)
demo.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://92bc4a46f66c01aef7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://92bc4a46f66c01aef7.gradio.live
